# Avaliação por Partidas Reais — CNN vs Minimax

Este notebook foi isolado para utilizar o ambiente `.venv_tf` (TensorFlow 2.21+), que possui compatibilidade com o modelo `.tflite` exportado no Google Colab (opcodes mais recentes).

**Instrução Importante:** Certifique-se de selecionar o Kernel `.venv_tf` (ou Python do `.venv_tf`) no VS Code antes de rodar a célula abaixo.

In [1]:
import sys, os, io
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath('../..'))

from tqdm.auto import tqdm

from gerador_dados.jogo_pontinhos.avaliador_partidas_pontinhos import (
    avaliar_paralelo, imprimir_relatorio,
    todos_labels_canonicos
)

# =========================================================================
# CONFIGURAÇÃO
# =========================================================================
CAMINHO_MODELO  = '../../modelos/pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss.tflite'
RELATORIO       = f'pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss_avaliacao'
TAMANHO         = 'pequeno'
N_PARTIDAS      = 200    # total por profundidade (100 CNN 1ª a jogar + 100 CNN 2ª a jogar)
PROFUNDIDADES   = [1, 3, 5, 6]
TIMER_MS        = 0      # 0 = sem limite | ex: 5 = Minimax limitado a 5ms/jogada
MAX_WORKERS     = 14

# Canais usados no treinamento deste modelo (para documentação do relatório)
# CANAIS_MODELO = [
#     'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita', 'caixa_fechada',
# ]

CANAIS_MODELO = [
    'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita',
    'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta',
    'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta',
    'paridade_cadeia_longa_impar'
]

SALVAR_CAIXAS_PERDIDAS = True
BASE_VIS = Path('../../visualizacoes/avaliacao_partidas')
EXEC_ID  = f"{Path(CAMINHO_MODELO).stem}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXEC_DIR = BASE_VIS / EXEC_ID

# =========================================================================
# CAPTURA DE RELATÓRIO
# =========================================================================
_report_buf = io.StringIO()

def rprint(*args, sep=' ', end='\n'):
    print(*args, sep=sep, end=end)
    msg = sep.join(str(a) for a in args) + end
    _report_buf.write(msg)

def _md_h(level, text):
    rprint('\n' + '#' * level + ' ' + text + '\n')

def _md_sep():
    rprint('\n---\n')

def _md_table_rows(rows, index_col=None):
    """Tabela markdown a partir de lista de dicts — sem pandas."""
    if not rows:
        return
    all_cols = list(rows[0].keys())
    if index_col and index_col in all_cols:
        all_cols.remove(index_col)
        header = [index_col] + all_cols
    else:
        header = all_cols
        index_col = None

    data = []
    for row in rows:
        data.append([str(row.get(c, '')) for c in header])

    widths = [len(h) for h in header]
    for row in data:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(cell))

    fmt = lambda cells: '| ' + ' | '.join(c.ljust(widths[i]) for i, c in enumerate(cells)) + ' |'
    rprint(fmt(header))
    rprint('| ' + ' | '.join('-' * w for w in widths) + ' |')
    for row in data:
        rprint(fmt(row))
    rprint()

# =========================================================================
# EXECUÇÃO
# =========================================================================
labels = todos_labels_canonicos(4, 3)

# Cabeçalho do relatório
_md_h(1, 'Relatório de Avaliação — CNN vs Minimax')
rprint('| Parâmetro | Valor |')
rprint('|-----------|-------|')
rprint(f'| Data | {datetime.now().strftime("%Y-%m-%d %H:%M")} |')
rprint(f'| Modelo | `{Path(CAMINHO_MODELO).name}` |')
rprint(f'| Canais ({len(CANAIS_MODELO)}) | {", ".join(CANAIS_MODELO)} |')
rprint(f'| Partidas por profundidade | {N_PARTIDAS} |')
rprint(f'| Profundidades | {PROFUNDIDADES} |')
timer_desc = f'{TIMER_MS}ms/jogada (iterative deepening)' if TIMER_MS > 0 else 'sem limite'
rprint(f'| Timer Minimax | {timer_desc} |')
rprint()
_md_sep()

stats_list = []
if __name__ == '__main__':
    if TIMER_MS > 0:
        print(f'Timer ativo: Minimax limitado a {TIMER_MS}ms por jogada')
    if SALVAR_CAIXAS_PERDIDAS:
        print(f'Salvando visualizações de caixas perdidas em: {EXEC_DIR}')

    for prof in PROFUNDIDADES:
        nome = f'Minimax(p<={prof}, {TIMER_MS}ms)' if TIMER_MS > 0 else f'Minimax(p={prof})'

        with tqdm(total=N_PARTIDAS, desc=f'{nome:25s}', unit='partida', leave=True) as pbar:
            c = [0, 0, 0, 0, 0]

            def _cb(completed, total, result, _pbar=pbar, _c=c):
                if result['vencedor'] == 1:   _c[0] += 1
                elif result['vencedor'] == 0:  _c[1] += 1
                else:                          _c[2] += 1
                _c[3] += result.get('opp_perdidas_a1', 0)
                _c[4] += result.get('opp_total_a1', 0)
                _pbar.update(completed - _pbar.n)
                postfix = {'V': _c[0], 'E': _c[1], 'D': _c[2]}
                if _c[4] > 0:
                    postfix['?caixa'] = f"{_c[3]}/{_c[4]}"
                _pbar.set_postfix(postfix)

            saida_misses = (EXEC_DIR / f'minimax_p{prof}') if SALVAR_CAIXAS_PERDIDAS else None

            s = avaliar_paralelo(
                CAMINHO_MODELO, labels, prof, nome, TAMANHO,
                N_PARTIDAS, TIMER_MS, MAX_WORKERS,
                progress_callback=_cb,
                salvar_caixas_perdidas_em=saida_misses,
            )
        stats_list.append(s)

    imprimir_relatorio(stats_list)

d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\.venv_tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



# Relatório de Avaliação — CNN vs Minimax

| Parâmetro | Valor |
|-----------|-------|
| Data | 2026-05-21 10:53 |
| Modelo | `pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss.tflite` |
| Canais (12) | aresta_topo, aresta_base, aresta_esquerda, aresta_direita, caixa_fechada, eh_grau3, eh_grau2, em_cadeia_curta, em_cadeia_longa, em_loop, em_cadeia_aberta_uma_ponta, paridade_cadeia_longa_impar |
| Partidas por profundidade | 200 |
| Profundidades | [1, 3, 5, 6] |
| Timer Minimax | sem limite |


---

Salvando visualizações de caixas perdidas em: ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss__20260521_105354


Minimax(p=1)             : 100%|██████████| 200/200 [00:50<00:00,  3.96partida/s, V=193, E=7, D=0, ?caixa=44/1981]


  Eventos de caixa perdida salvos: 44 -> ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss__20260521_105354\minimax_p1


Minimax(p=3)             : 100%|██████████| 200/200 [00:24<00:00,  8.27partida/s, V=164, E=21, D=15, ?caixa=77/1681]


  Eventos de caixa perdida salvos: 77 -> ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss__20260521_105354\minimax_p3


Minimax(p=5)             : 100%|██████████| 200/200 [05:36<00:00,  1.68s/partida, V=143, E=21, D=36, ?caixa=98/1550]


  Eventos de caixa perdida salvos: 98 -> ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss__20260521_105354\minimax_p5


Minimax(p=6)             : 100%|██████████| 200/200 [17:44<00:00,  5.32s/partida, V=127, E=20, D=53, ?caixa=90/1406]

  Eventos de caixa perdida salvos: 90 -> ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_cnn_depth_11_e_20_12canais_valloss__20260521_105354\minimax_p6

AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax

  Adversário: Minimax(p=1)  (200 partidas)
  Vitórias CNN          193  ( 96.5%)
  Empates                 7  (  3.5%)
  Derrotas CNN            0  (  0.0%)
  Tempo médio CNN:  0.27 ms/jogada
  Tempo médio Minimax(p=1): 0.2 ms/jogada
  CNN é 1× mais rápida
  Caixas deixadas p/ Minimax:   44 / 1981 oportunidades (2.2%)

  Adversário: Minimax(p=3)  (200 partidas)
  Vitórias CNN          164  ( 82.0%)
  Empates                21  ( 10.5%)
  Derrotas CNN           15  (  7.5%)
  Tempo médio CNN:  0.24 ms/jogada
  Tempo médio Minimax(p=3): 83.1 ms/jogada
  CNN é 346× mais rápida
  Caixas deixadas p/ Minimax:   77 / 1681 oportunidades (4.6%)

  Adversário: Minimax(p=5)  (200 partidas)
  Vitórias CNN          143  ( 71.5%)
  Empates                21  ( 10.5%)
  Derrotas CNN           36

In [2]:
# =========================================================================
# RELATÓRIO MARKDOWN — Resultados da Avaliação
# Executar após a célula principal. Salva .md na pasta do projeto.
# =========================================================================
_md_h(2, 'Resultados por Adversário')

# Tabela resumo
rows = []
for s in stats_list:
    fator = f"{s['fator_velocidade']:.0f}×"
    opp = '—'
    if s.get('opp_total_cnn', 0) > 0:
        opp_p = f"{s['opp_perdidas_cnn'] / s['opp_total_cnn'] * 100:.1f}%"
        opp = f"{s['opp_perdidas_cnn']}/{s['opp_total_cnn']} ({opp_p})"
    rows.append({
        'Adversário':      s['adversario'],
        'Partidas':        str(s['partidas']),
        'Vitórias CNN':    f"{s['vitorias_cnn']} ({s['pct_vitorias']:.1f}%)",
        'Empates':         f"{s['empates']} ({s['pct_empates']:.1f}%)",
        'Derrotas CNN':    f"{s['derrotas_cnn']} ({s['pct_derrotas']:.1f}%)",
        'T. CNN (ms)':     f"{s['tempo_cnn_ms']:.2f}",
        'T. MM (ms)':      f"{s['tempo_mm_ms']:.1f}",
        'CNN mais rápida': fator,
        'Caixas cedidas':  opp,
    })

_md_table_rows(rows, index_col='Adversário')

# Detalhes por adversário
_md_h(2, 'Detalhes por Adversário')
for s in stats_list:
    _md_h(3, s['adversario'])
    rprint('| Métrica | Valor |')
    rprint('|---------|-------|')
    rprint(f'| Vitórias CNN | {s["vitorias_cnn"]} ({s["pct_vitorias"]:.1f}%) |')
    rprint(f'| Empates | {s["empates"]} ({s["pct_empates"]:.1f}%) |')
    rprint(f'| Derrotas CNN | {s["derrotas_cnn"]} ({s["pct_derrotas"]:.1f}%) |')
    rprint(f'| Tempo médio CNN | {s["tempo_cnn_ms"]:.2f} ms/jogada |')
    rprint(f'| Tempo médio Minimax | {s["tempo_mm_ms"]:.1f} ms/jogada |')
    rprint(f'| CNN é mais rápida | {s["fator_velocidade"]:.0f}× |')
    if s.get('opp_total_cnn', 0) > 0:
        opp_pct = s['opp_perdidas_cnn'] / s['opp_total_cnn'] * 100
        rprint(f'| Caixas cedidas ao Minimax | {s["opp_perdidas_cnn"]}/{s["opp_total_cnn"]} ({opp_pct:.1f}%) |')
    if 'prof_media_mm' in s:
        rprint(f'| Profundidade média alcançada (MM) | {s["prof_media_mm"]:.1f} |')
    rprint()

_md_sep()

# Salvar relatório
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
nome_md = f"{RELATORIO}_{ts}.md"
caminho_md = Path('../../resultados/jogo_pontinhos') / nome_md
caminho_md.parent.mkdir(parents=True, exist_ok=True)

with open(caminho_md, 'w', encoding='utf-8') as f:
    f.write(_report_buf.getvalue())
print(f'Relatório salvo: {caminho_md} ({caminho_md.stat().st_size / 1024:.1f} KB)')


## Resultados por Adversário

| Adversário   | Partidas | Vitórias CNN | Empates    | Derrotas CNN | T. CNN (ms) | T. MM (ms) | CNN mais rápida | Caixas cedidas |
| ------------ | -------- | ------------ | ---------- | ------------ | ----------- | ---------- | --------------- | -------------- |
| Minimax(p=1) | 200      | 193 (96.5%)  | 7 (3.5%)   | 0 (0.0%)     | 0.27        | 0.2        | 1×              | 44/1981 (2.2%) |
| Minimax(p=3) | 200      | 164 (82.0%)  | 21 (10.5%) | 15 (7.5%)    | 0.24        | 83.1       | 346×            | 77/1681 (4.6%) |
| Minimax(p=5) | 200      | 143 (71.5%)  | 21 (10.5%) | 36 (18.0%)   | 0.26        | 1648.0     | 6363×           | 98/1550 (6.3%) |
| Minimax(p=6) | 200      | 127 (63.5%)  | 20 (10.0%) | 53 (26.5%)   | 0.26        | 5055.2     | 19364×          | 90/1406 (6.4%) |


## Detalhes por Adversário


### Minimax(p=1)

| Métrica | Valor |
|---------|-------|
| Vitórias CNN | 193 (96.5%) |
| Empates | 7 (3.5%) |
| Derrotas CNN | 0 (0.0%) |


## Resultado Profundidade 6

In [3]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# **CNN TREINADA COM TABULEIROS GERADOS DE FORMA ALEATÓRIA**
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          192  ( 96.0%)
#   Empates                 4  (  2.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.06 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 3× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          105  ( 52.5%)
#   Empates                26  ( 13.0%)
#   Derrotas CNN           69  ( 34.5%)
#   Tempo médio CNN:  0.10 ms/jogada
#   Tempo médio Minimax(p=3): 82.4 ms/jogada
#   CNN é 847× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           44  ( 22.0%)
#   Empates                13  (  6.5%)
#   Derrotas CNN          143  ( 71.5%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1639.1 ms/jogada
#   CNN é 12519× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN            3  (  1.5%)
#   Empates                10  (  5.0%)
#   Derrotas CNN          187  ( 93.5%)
#   Tempo médio CNN:  0.15 ms/jogada
#   Tempo médio Minimax(p=6): 5107.0 ms/jogada
#   CNN é 34794× mais rápida
# 
# ========================================================================

## Resultado Profundidade 7

In [4]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          184  ( 92.0%)
#   Empates                12  (  6.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.09 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 2× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          120  ( 60.0%)
#   Empates                54  ( 27.0%)
#   Derrotas CNN           26  ( 13.0%)
#   Tempo médio CNN:  0.11 ms/jogada
#   Tempo médio Minimax(p=3): 80.6 ms/jogada
#   CNN é 711× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           89  ( 44.5%)
#   Empates                59  ( 29.5%)
#   Derrotas CNN           52  ( 26.0%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1627.1 ms/jogada
#   CNN é 12419× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN           80  ( 40.0%)
#   Empates                46  ( 23.0%)
#   Derrotas CNN           74  ( 37.0%)
#   Tempo médio CNN:  0.14 ms/jogada
#   Tempo médio Minimax(p=6): 5820.3 ms/jogada
#   CNN é 41782× mais rápida
# 
# ========================================================================